# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task

### ML Task Type: Content Decay Prediction (Binary Classification)

- **Task type:** Binary classification, used as a ranking model (each page gets a decay score).
- **Question answered:** Which published pages lost 30% or more of their search impressions, so that they should be reviewed first for a content refresh?
- **Action supported:** The ranked list is a decision-support queue for the editorial and SEO team. An editor reviews the top-ranked pages; the model does not decide or rewrite anything on its own.

In [1]:
!test -d repo || git clone --depth 1 https://github.com/flyrank-bih/flyrank-ml-internship-starter.git repo
!cp -r repo/data .

Cloning into 'repo'...
remote: Enumerating objects: 94, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 94 (delta 12), reused 29 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (94/94), 1.83 MiB | 3.70 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, precision_recall_curve

SEED = 42
K = 500
DATA_PATH = "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Loaded:", df.shape)

Loaded: (30000, 44)


## 2. Target / Proxy

### Target: `is_decayed`

- **Type:** Binary classification (1 = decaying page, 0 = not decaying).
- **Proxy basis:** `trend_pct` in the starter dataset. Observed in the code cell below: `trend_pct` equals the percent change in impressions between the last 30 days and the prior 30 days (maximum difference 0.05, a rounding effect).
- **Rule:** `is_decayed = 1` if `trend_direction == 'down'` and `trend_pct <= -30%`; otherwise `0`. Label 0 therefore includes stable, growing, and mildly declining pages (a decline smaller than 30%).
- **Pages left out, step 1 (no baseline):** 3,388 of 30,000 pages (2,236 `new`, 1,152 `flat`) had zero impressions in the prior 30 days, so `trend_pct` is undefined. They are excluded instead of being silently labeled 0. This leaves 26,612 labeled pages.
- **Pages left out, step 2 (review population):** 4,854 labeled pages with fewer than 100 impressions in 90 days are excluded. For such pages the label is often forced by the data: a page whose whole 90-day total is 1 impression, and which is labeled at all, must have lost it in the last 30 days, so it counts as decayed. A model found this shortcut (shown in ML-08), so the floor was added after that check, not before. It matches the 100-impression floor of the Week-4 baseline.
- **Prevalence in the review population:** 21,758 pages; 11,174 decaying (51.4%) and 10,584 not decaying (48.6%).

### Rationale
A refresh is triggered by a clear drop in search visibility, and `trend_pct` records that drop. The -30% cutoff is an assumption chosen as a meaningful decline; it was not tuned, and other cutoffs were not tested.

### Leakage control
- `trend_direction` and `trend_pct` define the label and are excluded from the features.
- `impressions_last_30d` and `impressions_prev_30d` are the two inputs of `trend_pct`, so they are excluded: a model given both could recompute the label.
- The other 30-day window columns (clicks and sessions, last and prior 30 days) are also excluded. They measure the same recent period as the label, so keeping them would let the model detect decay that has already happened instead of ranking pages by their profile.
- `content_id` and `client_id` are identifiers and are not features. `client_id` is used only to group pages when splitting data.
- A guard (`assert`) in the modeling cell fails if any excluded column enters the feature set.

### Limitations
- The label describes **current decline** from a single snapshot. It helps prioritize pages that are already declining; it does not forecast future decay.
- `trend_pct` is not seasonally adjusted, so some declines may be seasonal.
- Even above the 100-impression floor, a -30% change on a page with a few hundred impressions can be noise. The floor and the 30% cutoff are assumptions and were not tuned.
- Results apply to pages that have a prior-period baseline and at least 100 impressions in 90 days; smaller pages are not covered.

In [3]:
DECAY_THRESHOLD = -30.0  # percent change in impressions, last 30 days vs prior 30 days
MIN_IMPRESSIONS = 100    # review population: enough traffic to judge a decline

# trend_pct is undefined when the prior 30 days had zero impressions (no baseline).
# Those pages cannot be judged as decaying or not, so they are left out of labeling.
no_baseline = df["trend_pct"].isna()
print("Pages without a baseline (excluded):", int(no_baseline.sum()))
print(df.loc[no_baseline, "trend_direction"].value_counts().to_string())
df = df.loc[~no_baseline].reset_index(drop=True)
n_labeled = len(df)

# Target: 1 = fell by 30% or more, 0 = everything else (stable, growing, or smaller decline)
df["is_decayed"] = ((df["trend_direction"] == "down") &
                    (df["trend_pct"] <= DECAY_THRESHOLD)).astype(int)

# Leakage check: trend_pct IS the impressions change, so those two columns define the label
chg = (df["impressions_last_30d"] - df["impressions_prev_30d"]) / df["impressions_prev_30d"] * 100
print("\nMax |trend_pct - impressions change|:", round((chg - df["trend_pct"]).abs().max(), 3))

# Review population: below this volume the label is often forced by the data (see ML-08)
df = df[df["impressions_90d"] >= MIN_IMPRESSIONS].reset_index(drop=True)
print(f"\nLabeled pages: {n_labeled:,} | review population (>= {MIN_IMPRESSIONS} impressions in 90 days): {len(df):,}")

# Columns that define the label (excluded from all models)
LABEL_SOURCES = ["trend_direction", "trend_pct"]
LEAKY = ["impressions_last_30d", "impressions_prev_30d"]

print(pd.DataFrame({"count": df["is_decayed"].value_counts(),
                    "share": df["is_decayed"].value_counts(normalize=True).round(3)}))
print("\nLeakage exclusions:", LABEL_SOURCES + LEAKY)

Pages without a baseline (excluded): 3388
trend_direction
new     2236
flat    1152

Max |trend_pct - impressions change|: 0.05

Labeled pages: 26,612 | review population (>= 100 impressions in 90 days): 21,758
            count  share
is_decayed              
1           11174  0.514
0           10584  0.486

Leakage exclusions: ['trend_direction', 'trend_pct', 'impressions_last_30d', 'impressions_prev_30d']


## 3. Success Metric

### Primary Metric: Precision@K

- **Definition:** Precision@K is the share of the K highest-ranked pages that are truly decaying (`is_decayed = 1`).
- **K = 500:** an assumed editorial capacity per cycle. The dataset has no capacity field, so this value should be replaced with the team's real number.
- **Target (set before any model was run):** Precision@500 ≥ 0.80 on clients not seen in training (group-aware split on `client_id`).
- **Baseline:** 51.4% of pages in the review population are decaying, so random ranking gives Precision@K ≈ 0.51. A target of 0.80 is a 1.56x lift; the maximum possible lift is 1.95x.
- **Observed (Section 5 cell):** pooled out-of-fold Precision@500 = 0.710 (1.38x lift); per-fold Precision@500 = 0.717 ± 0.090. Pooled Precision@3,000 = 0.694.
- **Status:** the target is **not met** on either view. It is kept as originally set and was not lowered after seeing results. An earlier version of this notebook reported 0.866; that figure included pages with fewer than 100 impressions, whose labels are forced by the data, and it is not used.

### Secondary Metric: Recall (at an operating threshold, not at K)

- **Definition:** Recall is the share of all decaying pages that the model flags.
- **Target:** Recall ≥ 0.75.
- **Why not Recall@K:** with 11,174 decaying pages, Recall@500 cannot exceed 4.5% and Recall@2,175 (10% of pages) cannot exceed 19.5%. Reaching 0.75 recall needs at least 8,381 pages flagged, so recall only makes sense at a threshold.
- **Observed:** at Recall ≥ 0.75 the model's Precision is 0.613. Precision stays ≥ 0.60 up to Recall 0.783, and ≥ 0.70 only up to Recall 0.097.
- **Implication:** the model supports a modestly precise top of the queue, but a high-recall sweep of all decaying pages has low precision. The two original targets (Precision ≥ 0.80 and Recall ≥ 0.75 together) are not reached at the same operating point.

### Why this fits the business action
Editor time is limited. Precision@K measures how much of that time goes to genuinely declining pages instead of false alarms. Recall shows how many declining pages the queue does not reach.

### Comparison baseline
Model results are compared with three simple rules (worst `avg_position`, lowest `ctr`, and an if-statement on both), shown as lift in Precision@500.

In [4]:
N = len(df)
P = int(df["is_decayed"].sum())
base_rate = P / N

print(f"Pages: {N} | Decaying: {P} | Base rate: {base_rate:.3f}")
print(f"Max possible lift: {1 / base_rate:.2f}x")
for k in [K, int(0.10 * N)]:
    print(f"K = {k}: max Recall@K = {min(k, P) / P:.3f}")
print(f"K needed for Recall 0.75 at perfect precision: {int(np.ceil(0.75 * P))}")

Pages: 21758 | Decaying: 11174 | Base rate: 0.514
Max possible lift: 1.95x
K = 500: max Recall@K = 0.045
K = 2175: max Recall@K = 0.195
K needed for Recall 0.75 at perfect precision: 8381


## 4. Unit of Analysis

**One row = one content page**, identified by `content_id`, owned by one client (`client_id`), observed at a single point-in-time snapshot.

| Item | Definition |
|---|---|
| **Grain** | One row per `content_id`: 30,000 pages loaded (44 source columns), 26,612 labeled, and 21,758 in the review population used for modeling |
| **Key** | `content_id` (unique, 0 duplicates confirmed in the code cell) |
| **Grouping** | `client_id`: 30 clients in the review population; median 250 pages per client; the largest client holds 30.2% of rows. Used for the group-aware split |
| **Target** | `is_decayed` (1 = fell 30% or more in impressions, 0 = not) |
| **Features (34)** | Page attributes (`content_type`, `main_intent`, `word_count`, `char_count`, tiers), keyword demand (`search_volume`, `competition`, `cpc`), 90-day traffic and engagement totals, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, content age and freshness |
| **Excluded fields** | Label columns (`trend_direction`, `trend_pct`); label inputs and all other 30-day window columns; identifiers (`content_id`, `client_id`) |
| **Excluded rows** | 3,388 pages with no prior-period baseline, and 4,854 labeled pages with fewer than 100 impressions in 90 days (section 2) |

**Data notes:**
- `word_count` is missing for 30.0% of pages in the review population. The model handles missing values natively, so no imputation was applied.
- Decay rate differs a lot between clients (min 0.00, median 0.51, max 1.00 in the code cell), which is why the train/test split is grouped by client.

**Prediction output:** a decay score per page, ranked across the review population, from which the top K = 500 pages go to the editorial queue.

In [5]:
unit_cols = ["content_id", "content_type", "main_intent", "word_count",
             "search_volume", "avg_position", "ctr", "is_decayed"]

pages_per_client = df["client_id"].value_counts()
client_decay = df.groupby("client_id")["is_decayed"].mean()

print("Rows:", len(df), "| Unique content_id:", df["content_id"].nunique())
print("Clients:", len(pages_per_client),
      "| median pages/client:", int(pages_per_client.median()),
      "| largest client share:", f"{pages_per_client.iloc[0] / len(df):.1%}")
print(f"Decay rate by client: min {client_decay.min():.2f}, "
      f"median {client_decay.median():.2f}, max {client_decay.max():.2f}")
print(f"word_count missing: {df['word_count'].isna().mean():.1%} of pages")
df[unit_cols].head(5)

Rows: 21758 | Unique content_id: 21758
Clients: 30 | median pages/client: 250 | largest client share: 30.2%
Decay rate by client: min 0.00, median 0.51, max 1.00
word_count missing: 30.0% of pages


,content_id,content_type,main_intent,word_count,search_volume,avg_position,ctr,is_decayed
0,content_304f48230142,keyword article,transactional,3221.0,10.0,10.6,0.76,1
1,content_a1fb4e703a9e,keyword article,informational,2481.0,90.0,20.3,0.05,1
2,content_9aa793d4d895,keyword article,informational,3515.0,0.0,36.5,0.09,1
3,content_331d6c4de07b,keyword article,commercial,NaN,10.0,6.2,0.49,0
4,content_d99b7a2d90ca,keyword article,informational,2803.0,0.0,44.0,0.13,1


## 5. Why ML Beats a Fixed Rule

### Evidence (out-of-fold, 21,758 pages, base rate 0.514)

Every page is scored by a model that never saw its client (5-fold `GroupKFold` on `client_id`). Rules need no training and are scored on the same pages.

| Method | Precision@500 | AUC | Lift vs. base |
|---|---|---|---|
| Rule: worst `avg_position` | 0.292 | 0.474 | 0.57x |
| Rule: lowest `ctr` | 0.578 | 0.572 | 1.13x |
| Rule: `avg_position > 10` AND `ctr < 0.5` (precision of all flagged pages) | 0.518 | 0.505 | 1.01x |
| **ML: gradient boosting** | **0.710** | **0.667** | **1.38x** |

- Per-fold results for the model: AUC 0.654 ± 0.026 and Precision@500 0.717 ± 0.090.
- The if-statement flags 11,307 pages (52% of the review population) with recall 0.524. It cannot rank pages inside the flagged group, so it cannot fill a 500-page queue by priority; the 0.518 shown is the expected precision of 500 pages picked at random from the flagged set.
- The best single rule (lowest `ctr`) reaches 1.13x lift; the ML model reaches 1.38x at the top of the queue.

### Why ML fits this task
- **Ranking:** editors can review about 500 pages per cycle, so the output must be an ordered list. A binary rule cannot order pages.
- **Many weak signals:** a one- or two-column rule with hand-picked cutoffs was weak (AUC 0.47 to 0.57); combining many features gave AUC 0.67.
- **Missing data:** gradient boosting handles missing values (30% of `word_count`) without manual rules.
- **Unseen clients:** evaluation uses clients absent from training, so the result reflects patterns that transfer between clients.

### Limitations
- The gain is moderate, not strong: AUC is 0.67 and lift is about 1.4x. Results are decision-support, not a guarantee, and the Precision@500 target of 0.80 was not met (section 3).
- Precision@500 depends on how it is measured: 0.710 pooled (500 of 21,758 pages) versus 0.717 ± 0.090 per fold (500 of about 4,350 pages per fold on average). The spread across folds is wide. Pooled scores also come from five different fold models.
- Decay rate varies from 0.00 to 1.00 across clients, so part of the lift may come from client-level patterns. A new client's decay rate is unknown.
- The rules are untuned, hand-picked thresholds with directions fixed in advance; a tuned rule might do better.
- The 90-day totals, `ctr`, `avg_position` and the tier columns are built from 90-day data, which overlaps the 30-day windows that define the label. In ML-08, the same kind of model using only page attributes fell to AUC 0.59 and Precision@500 0.48, so much of this result reflects declines that are already visible in recent activity.
- K = 500 is an assumed capacity. The label limits in Section 2 also apply.

In [6]:
# ---- Features: drop ids, label, label sources, known leaks, and all 30-day window columns
WINDOWS = [c for c in df.columns if c.endswith(("_last_30d", "_prev_30d"))]
drop = {"content_id", "client_id", "is_decayed"} | set(LABEL_SOURCES) | set(LEAKY) | set(WINDOWS)
features = [c for c in df.columns if c not in drop]

X = df[features].copy()
for c in X.select_dtypes(exclude="number").columns:
    X[c] = X[c].astype("category")
y = df["is_decayed"].values
groups = df["client_id"].values

assert not set(X.columns) & (set(LABEL_SOURCES) | set(LEAKY) | set(WINDOWS)), "Leakage detected"
print(f"Features used: {X.shape[1]}")

def p_at_k(y_true, score, k=K):
    return y_true[np.argsort(-score, kind="stable")[:k]].mean()

# ---- ML: 5-fold out-of-fold scores; a client is never in both train and test
oof = np.zeros(len(y))
fold_auc, fold_p = [], []
for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
    model = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                           categorical_features="from_dtype",
                                           random_state=SEED)
    model.fit(X.iloc[tr], y[tr])
    oof[te] = model.predict_proba(X.iloc[te])[:, 1]
    fold_auc.append(roc_auc_score(y[te], oof[te]))
    fold_p.append(p_at_k(y[te], oof[te]))

# ---- Fixed rules (no training), scored on the same pages
pos = df["avg_position"].fillna(df["avg_position"].median()).values
ctr = df["ctr"].fillna(df["ctr"].median()).values
flag = (pos > 10) & (ctr < 0.5)

rows = [
    ("Rule: worst avg_position",                   p_at_k(y, pos),  roc_auc_score(y, pos)),
    ("Rule: lowest ctr",                           p_at_k(y, -ctr), roc_auc_score(y, -ctr)),
    ("Rule: position>10 AND ctr<0.5 (all flagged)", y[flag].mean(),  roc_auc_score(y, flag.astype(int))),
    ("ML: gradient boosting",                      p_at_k(y, oof),  roc_auc_score(y, oof)),
]
res = pd.DataFrame(rows, columns=["method", f"precision@{K}", "auc"]).round(3)
res["lift_vs_base"] = (res[f"precision@{K}"] / y.mean()).round(2)
print(f"\nPooled out-of-fold results ({len(y)} pages, base rate {y.mean():.3f}):")
print(res.to_string(index=False))
print(f"\nML per-fold AUC:          {np.mean(fold_auc):.3f} ± {np.std(fold_auc):.3f}")
print(f"ML per-fold Precision@{K}: {np.mean(fold_p):.3f} ± {np.std(fold_p):.3f}")
print(f"If-statement flags {flag.sum()} pages ({flag.mean():.0%}); "
      f"recall {y[flag].sum() / y.sum():.3f}")

# ---- Operating-point metrics for the ML model
prec, rec, _ = precision_recall_curve(y, oof)
i = np.where(rec >= 0.75)[0][-1]
print(f"\nPrecision at Recall >= 0.75: {prec[i]:.3f}")
for t in [0.60, 0.70]:
    print(f"Max Recall at Precision >= {t}: {rec[prec >= t].max():.3f}")
print(f"Pooled Precision@3000: {p_at_k(y, oof, 3000):.3f}")

Features used: 34

Pooled out-of-fold results (21758 pages, base rate 0.514):
                                     method  precision@500   auc  lift_vs_base
                   Rule: worst avg_position          0.292 0.474          0.57
                           Rule: lowest ctr          0.578 0.572          1.13
Rule: position>10 AND ctr<0.5 (all flagged)          0.518 0.505          1.01
                      ML: gradient boosting          0.710 0.667          1.38

ML per-fold AUC:          0.654 ± 0.026
ML per-fold Precision@500: 0.717 ± 0.090
If-statement flags 11307 pages (52%); recall 0.524

Precision at Recall >= 0.75: 0.613
Max Recall at Precision >= 0.6: 0.783
Max Recall at Precision >= 0.7: 0.097
Pooled Precision@3000: 0.694


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.